# Stereo Visual Odometry Example (Large Dataset)

Similar to the notebook `StereoVOExample.ipynb`, this notebook demonstrates a 3D stereo visual odometry problem using the GTSAM Python wrapper. The main difference is that this example uses a larger dataset; in other words, we take on a less trivial setting where the robot must observe multiple landmarks as it moves forward for a longer distance. The scenario is as follows:

1.  A robot starts at the origin (Pose 1: `X(1)`).
2.  The robot moves forward, taking periodic stereo measurements.
3.  The robot observes multiple landmarks with a stereo camera.

We will:
*   Define camera calibration and noise models.
*   Create a factor graph representing the problem.
*   Provide initial estimates for poses and landmark positions.
*   Optimize the graph using Levenberg-Marquardt to find the most probable configuration.

GTSAM Copyright 2010-2025, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/python/gtsam/examples/StereoVOExample_large.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install necessary libraries
%pip install -q gtsam-develop
%pip install -q numpy
%pip install -q matplotlib

In [1]:
import numpy as np
from pathlib import Path
import gtsam
from gtsam.utils.plot import plot_pose3
import matplotlib.pyplot as plt
import gtsam.utils.plot as gtsam_plot

# For shorthand for common GTSAM types (like X for poses, L for landmarks)
from gtsam.symbol_shorthand import X, L

## 1. Setup Factor Graph

We start by creating an empty `NonlinearFactorGraph`.

In [18]:
# Create an empty nonlinear factor graph
graph = gtsam.NonlinearFactorGraph()

## 2. Define Camera Calibration and Measurement Noise
- **Noise Model:** We define an isotropic noise model for the stereo measurements. `Isotropic.Sigma(3, 1.0)` means a 3-dimensional noise (for $u_L, u_R, v$) with a standard deviation of 1 pixel for each dimension.
- **Camera Calibration ([`Cal3_S2Stereo`](/gtsam/geometry/doc/Cal3_S2Stereo.ipynb)):** We setup a stereo calibration model with the following properties, which are read from a calibration text file:
  - $f_x=f_y=721.5377$
  - $s=0$
  - $(u,v)=(609.5593,\,172.854)$
  - $b=0.537150588$

In [21]:
# Calibration file
CALIBRATION_FILE_PATH = Path.cwd() / "Data/VO_calibration.txt"

# Read calibration parameters
cal_params = np.loadtxt(CALIBRATION_FILE_PATH)

# Create a Cal3_S2Stereo calibration object
K = gtsam.Cal3_S2Stereo(cal_params)
print(K)

# Define the stereo measurement noise model (isotropic, 1 pixel standard deviation)
measurement_noise = gtsam.noiseModel.Isotropic.Sigma(3, 1.0)

K: 721.538       0 609.559
      0 721.538 172.854
      0       0       1
Baseline: 0.537151



## 3. Read Camera Pose Parameters to make Initial Camera Pose Estimates 

In [ ]:
# Odometry data file with camera poses
ODOMETRY_DATA_FILE = Path.cwd() / "Data/VO_camera_poses_large.txt"

# Read poses
camera_poses = np.loadtxt(ODOMETRY_DATA_FILE)

# Create a gtsam.Values object to hold the pose data
initial_estimates = gtsam.Values()

# Parse the raw camera pose data
for pose in camera_poses:
	pose_id = int(pose[0])
	m = pose[1:].copy().reshape(4, 4)		# 16 data values reshaped to a 4x4 matrix with row major order
	initial_estimates.insert(X(pose_id), m)

print(initial_estimates)

In [ ]:
STEREO_FACTORS_FILE = "./examples/Data/VO_stereo_factors_large.txt"
ODOMETRY_FILE = "./examples/Data/VO_camera_poses_large.txt"

In [ ]:
# Create a NonlinearFactorGraph
graph = gtsam.NonlinearFactorGraph()

# Add a prior on the first pose, setting it to the origin
priorNoise = gtsam.noiseModel.Diagonal.Sigmas(np.array([0.1, 0.1, 0.1, 0.3, 0.3, 0.3]))
first_pose = gtsam.Pose3()
graph.add(gtsam.PriorFactorPose3(0, first_pose, priorNoise))

# Read stereo factors and add them to the graph
with open(STEREO_FACTORS_FILE, 'r') as f:
    for line in f:
        data = list(map(float, line.split()))
        j = int(data[-1])
        i = int(data[0])
        measured_pt = gtsam.StereoPoint2(data[1], data[2], data[3])
        stereo_noise = gtsam.noiseModel.Isotropic.Sigma(3, 1.0)
        graph.add(gtsam.GenericStereoFactor3D(measured_pt, stereo_noise, i, j, calibration))

# Read odometry measurements and add them to the graph
with open(ODOMETRY_FILE, 'r') as f:
    for line in f:
        data = list(map(float, line.split()))
        i, j = int(data[0]), int(data[1])
        odometry = gtsam.Pose3(np.array(data[2:]).reshape(4,4))
        odometry_noise = gtsam.noiseModel.Diagonal.Sigmas(
            np.array([0.1, 0.1, 0.1, 0.3, 0.3, 0.3]))
        graph.add(gtsam.BetweenFactorPose3(i, j, odometry, odometry_noise))

print(f"Graph size: {graph.size()} factors")

In [ ]:
# Initialize the values for the optimization
initial_estimate = gtsam.Values()
with open(ODOMETRY_FILE, 'r') as f:
    for line in f:
        data = list(map(float, line.split()))
        i = int(data[0])
        if not initial_estimate.exists(i):
            pose = gtsam.Pose3(np.array(data[2:8]).reshape(4,4)) if i ==0 else gtsam.Pose3(np.array(data[8:]).reshape(4,4))
            initial_estimate.insert(i, pose)

# Estimate the landmark initial values
with open(STEREO_FACTORS_FILE, 'r') as f:
    for line in f:
        data = list(map(float, line.split()))
        j = int(data[-1])
        if not initial_estimate.exists(j):
            pose_key = int(data[0])
            stereo_pt = gtsam.StereoPoint2(data[1], data[2], data[3])
            pose = initial_estimate.atPose3(pose_key)
            camera = gtsam.PinholeStereoCamera(pose, calibration)
            estimated_landmark = camera.backproject(stereo_pt)
            initial_estimate.insert(j, estimated_landmark)

In [ ]:
# Print the initial error
initial_error = graph.error(initial_estimate)
print(f"Initial error: {initial_error}")

# Optimize the graph
optimizer = gtsam.LevenbergMarquardtOptimizer(graph, initial_estimate)
result = optimizer.optimize()

print("Optimization complete")

# Get the final error
final_error = graph.error(result)
print(f"Final error: {final_error}")

In [ ]:
# Plot the results
result_poses = gtsam.utilities.allPose3s(result)
fig = plt.figure(0)
ax = fig.add_subplot(111, projection='3d')

for i in range(result_poses.size()):
    gtsam_plot.plot_pose3(0, result_poses.atPose3(i), 10, ax=ax)

# Plot the landmarks
result_landmarks = gtsam.utilities.allPoints3(result)
for i in range(result_landmarks.size()):
    ax.scatter(result_landmarks.atPoint3(i).x(), result_landmarks.atPoint3(i).y(),
               result_landmarks.atPoint3(i).z(), c='b', marker='.')

ax.set_xlabel('X axis')
ax.set_ylabel('Y axis')
ax.set_zlabel('Z axis')
plt.show()